In [2]:
# Install required libraries

# Import libraries
import pandas as pd
import jieba
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score, classification_report
import matplotlib.pyplot as plt
import numpy as np
from sklearn.feature_selection import chi2
from collections import Counter
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from scipy.sparse import csr_matrix

# Import models from your friend's file
from models_copy import CentroidClassifier, KNeighborsClassifier, MultinomialNB, SVC, BalancedWinnow
from pynlpir import nlpir 
import pynlpir
import random

random.seed(42)
np.random.seed(42)

pynlpir.open()


# Load data
splits = {
    'train': 'data/train-00000-of-00001-02f200ca5f2a7868.parquet',
    'validation': 'data/validation-00000-of-00001-405befbaa3bcf1a2.parquet',
    'test': 'data/test-00000-of-00001-5372924f059fe767.parquet'
}

df = pd.read_parquet("hf://datasets/lansinuote/ChnSentiCorp/" + splits["train"])
val_df = pd.read_parquet("hf://datasets/lansinuote/ChnSentiCorp/" + splits["validation"])
test_df = pd.read_parquet("hf://datasets/lansinuote/ChnSentiCorp/" + splits["test"])

d:\anaconda3\envs\project7008\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df['cut_text'] = df['text'].apply(lambda x: ' '.join(pynlpir.segment(x, pos_tagging=False)))
val_df['val_cut_text'] = val_df['text'].apply(lambda x: ' '.join(pynlpir.segment(x, pos_tagging=False)))
test_df['test_cut_text'] = test_df['text'].apply(lambda x: ' '.join(pynlpir.segment(x, pos_tagging=False)))

# Vectorize text
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['cut_text'])
X_val = vectorizer.transform(val_df['val_cut_text'])
X_test = vectorizer.transform(test_df['test_cut_text'])

y = df['label']
y_val = val_df['label']
y_test = test_df['label']

In [4]:

# Custom function to calculate Mutual Information (MI) for all features
def calculate_mi_vectorized(X, y, class_labels):
    """
    Calculate Mutual Information (MI) for all features using vectorized operations.
    """
    # Convert y to a binary matrix (one-hot encoding)
    y_binary = np.array([y == c for c in class_labels]).T  # Shape: (n_samples, n_classes)

    # Compute co-occurrence counts
    A = X.T.dot(y_binary)  # A: t and c_i co-occur (shape: n_features x n_classes)
    B = X.sum(axis=0).reshape(-1, 1) - A  # B: t occurs without c_i
    E = y_binary.sum(axis=0) - A  # E: c_i occurs without t
    D = X.shape[0] - (A + B + E)  # D: neither t nor c_i occurs

    # Avoid division by zero
    A = np.where(A == 0, 1, A)  # Replace 0 with 1 to avoid log(0)
    B = np.where(B == 0, 1, B)
    E = np.where(E == 0, 1, E)
    D = np.where(D == 0, 1, D)

    # Calculate MI for each feature and class
    N = X.shape[0]
    mi_scores = np.log((A * N) / ((A + E) * (A + B)))  # Shape: n_features x n_classes

    # Return the maximum MI score across all classes for each feature
    return np.max(mi_scores, axis=1)

def select_features_mi(X, y, feature_names, k=10000):
    """
    Select top k features based on Mutual Information (MI).
    """
    # Get unique class labels
    class_labels = np.unique(y)

    # Calculate MI scores for all features
    mi_scores = calculate_mi_vectorized(X, y, class_labels)

    # Create a dictionary of feature names and their MI scores
    mi_scores_dict = {word: score for word, score in zip(feature_names, mi_scores)}
    sorted_mi_scores = sorted(mi_scores_dict.items(), key=lambda x: x[1], reverse=True)
    top_k_features = [word for word, score in sorted_mi_scores[:k]]
    return top_k_features

# Custom function to calculate Information Gain (IG)
def information_gain(X, y, feature_index):
    """
    Calculate Information Gain (IG) for a given feature.
    """
    # Calculate entropy of the target variable
    def entropy(y):
        _, counts = np.unique(y, return_counts=True)
        probabilities = counts / len(y)
        return -np.sum(probabilities * np.log2(probabilities + 1e-10))  # Add small epsilon to avoid log(0)

    # Calculate entropy of the feature
    def conditional_entropy(X_feature, y):
        unique_values = np.unique(X_feature)
        conditional_entropy = 0
        for value in unique_values:
            subset_indices = X_feature == value
            subset_y = y[subset_indices]
            if len(subset_y) > 0:
                conditional_entropy += (len(subset_y) / len(y)) * entropy(subset_y)
        return conditional_entropy

    # Calculate information gain
    total_entropy = entropy(y)
    conditional_entropy_value = conditional_entropy(X[:, feature_index], y)
    information_gain = total_entropy - conditional_entropy_value
    return information_gain

def select_features_ig(X, y, feature_names, k=10000):
    """
    Select top k features based on Information Gain (IG).
    """
    # Create a dictionary to store feature names and their indices
    feature_indices = {feature: index for index, feature in enumerate(feature_names)}

    # Convert sparse matrix to dense array
    X = X.toarray()

    # Calculate IG scores
    ig_scores = []
    for feature in feature_names:
        feature_index = feature_indices[feature]
        ig_score = information_gain(X, y, feature_index)
        ig_scores.append(ig_score)

    # Create a dictionary of feature names and their IG scores
    ig_scores_dict = {word: score for word, score in zip(feature_names, ig_scores)}
    sorted_ig_scores = sorted(ig_scores_dict.items(), key=lambda x: x[1], reverse=True)
    top_k_features = [word for word, score in sorted_ig_scores[:k]]
    return top_k_features

def calculate_chi_vectorized(X, y, class_labels):
    """
    Calculate Mutual Information (MI) for all features using vectorized operations.
    """
    # Convert y to a binary matrix (one-hot encoding)
    y_binary = np.array([y == c for c in class_labels]).T  # Shape: (n_samples, n_classes)

    # Compute co-occurrence counts
    A = X.T.dot(y_binary)  # A: t and c_i co-occur (shape: n_features x n_classes)
    B = X.sum(axis=0).reshape(-1, 1) - A  # B: t occurs without c_i
    E = y_binary.sum(axis=0) - A  # E: c_i occurs without t
    D = X.shape[0] - (A + B + E)  # D: neither t nor c_i occurs

    # Avoid division by zero
    A = np.where(A == 0, 1, A)  # Replace 0 with 1 to avoid log(0)
    B = np.where(B == 0, 1, B)
    E = np.where(E == 0, 1, E)
    D = np.where(D == 0, 1, D)

    # Calculate MI for each feature and class
    N = X.shape[0]
    chi_scores = N*np.square(A*D-B*E)/((A+E)*(B+D)*(A+B)*(E+D))

    # Return the maximum MI score across all classes for each feature
    return np.max(chi_scores, axis=1)


def select_features_chi(X, y, feature_names, k=10000):
    """
    Select top k features based on Chi-Square (CHI).
    """
    # Get unique class labels
    class_labels = np.unique(y)

    # Calculate MI scores for all features
    chi_scores = calculate_chi_vectorized(X, y, class_labels)
    
    chi_scores_dict = {word: score for word, score in zip(feature_names, chi_scores)}
    sorted_chi_scores = sorted(chi_scores_dict.items(), key=lambda x: x[1], reverse=True)
    top_k_features = [word for word, score in sorted_chi_scores[:k]]
    return top_k_features

def select_features_df(X, feature_names, k=10000):
    """
    Select top k features based on Document Frequency (DF).
    """
    df_scores = np.sum(X.toarray() > 0, axis=0)
    df_dict = dict(zip(feature_names, df_scores))
    counter = Counter(df_dict)
    top_k_features = [word for word, freq in counter.most_common(k)]
    return top_k_features

In [5]:

# Get feature names
feature_names = vectorizer.get_feature_names_out()

# Select features using MI, IG, CHI, and DF
top_k_mi = select_features_mi(X, y, feature_names, k=7000)  # Use optimized MI calculation
top_k_ig = select_features_ig(X, y, feature_names, k=7000)
top_k_chi = select_features_chi(X, y, feature_names, k=7000)
top_k_df = select_features_df(X, feature_names, k=7000)

# Create feature sets
feature_sets = {
    'MI': (TfidfVectorizer(vocabulary=top_k_mi), top_k_mi),
    'IG': (TfidfVectorizer(vocabulary=top_k_ig), top_k_ig),
    'CHI': (TfidfVectorizer(vocabulary=top_k_chi), top_k_chi),
    'DF': (TfidfVectorizer(vocabulary=top_k_df), top_k_df)
}

# Transform data using selected features
X_train_mi = feature_sets['MI'][0].fit_transform(df['cut_text'])
X_test_mi = feature_sets['MI'][0].transform(test_df['test_cut_text'])

X_train_ig = feature_sets['IG'][0].fit_transform(df['cut_text'])
X_test_ig = feature_sets['IG'][0].transform(test_df['test_cut_text'])

X_train_chi = feature_sets['CHI'][0].fit_transform(df['cut_text'])
X_test_chi = feature_sets['CHI'][0].transform(test_df['test_cut_text'])

X_train_df = feature_sets['DF'][0].fit_transform(df['cut_text'])
X_test_df = feature_sets['DF'][0].transform(test_df['test_cut_text'])

# Define the classifiers
classifiers = {
    'Centroid': CentroidClassifier(),
    'KNN': KNeighborsClassifier(),
    'NB': MultinomialNB(),
    'Winnow': BalancedWinnow(),
    'SVM': SVC(kernel='linear')
}

# Initialize lists to store F1 scores
micro_f1_scores = {clf: {fs: [] for fs in feature_sets} for clf in classifiers}
macro_f1_scores = {clf: {fs: [] for fs in feature_sets} for clf in classifiers}

# Function to find the best k for KNN
def find_best_k(X_train, y_train, X_val, y_val, k_range=range(1, 40)):
    accuracy = []
    for k in k_range:
        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(X_train, y_train)
        y_pred = knn.predict(X_val)
        accuracy.append(accuracy_score(y_val, y_pred))

    # Find the best k
    best_k = k_range[np.argmax(accuracy)]
    print(f"Best k: {best_k}, Max Accuracy: {max(accuracy)}")
    return best_k

# Find the best k for KNN for each feature set
best_k_values = {}
for fs_name, (vectorizer, features) in feature_sets.items():
    print(f"Finding best k for {fs_name} features...")
    # Transform the data using the current feature set
    X_train_fs = vectorizer.fit_transform(df['cut_text'])
    X_test_fs = vectorizer.transform(test_df['test_cut_text'])
    # Find the best k
    best_k = find_best_k(X_train_fs, y, X_test_fs, y_test)
    best_k_values[fs_name] = best_k

# Iterate over the number of features
num_features = 7000
for clf_name, clf in classifiers.items():
    for fs_name, (vectorizer, features) in feature_sets.items():
        # Select the top n features
        selected_features = features[:num_features]
        filtered_vectorizer = TfidfVectorizer(vocabulary=selected_features)
        X_train_fs = filtered_vectorizer.fit_transform(df['cut_text'])
        X_test_fs = filtered_vectorizer.transform(test_df['test_cut_text'])

        # Convert sparse matrices to dense matrices for classifiers that don't support sparse inputs
        if clf_name in ['Centroid', 'Winnow']:
            X_train_fs_dense = X_train_fs.toarray()
            X_test_fs_dense = X_test_fs.toarray()
        else:
            X_train_fs_dense = X_train_fs
            X_test_fs_dense = X_test_fs

        # For KNN, use the best k value found for the current feature set
        if clf_name == 'KNN':
            clf = KNeighborsClassifier(n_neighbors=best_k_values[fs_name])

        # Train the classifier on the subset of features
        clf.fit(X_train_fs_dense, y)
        # Predict on the test set
        y_pred = clf.predict(X_test_fs_dense)
        # Calculate micro and macro F1 scores
        micro_f1 = f1_score(y_test, y_pred, average='micro')
        macro_f1 = f1_score(y_test, y_pred, average='macro')
        # Store the scores
        micro_f1_scores[clf_name][fs_name].append(micro_f1)
        macro_f1_scores[clf_name][fs_name].append(macro_f1)


Finding best k for MI features...
Best k: 1, Max Accuracy: 0.7408333333333333
Finding best k for IG features...
Best k: 31, Max Accuracy: 0.8141666666666667
Finding best k for CHI features...
Best k: 37, Max Accuracy: 0.8058333333333333
Finding best k for DF features...
Best k: 25, Max Accuracy: 0.8133333333333334


In [6]:
micro_f1_table = pd.DataFrame(micro_f1_scores)
macro_f1_table = pd.DataFrame(macro_f1_scores)
print(micro_f1_table)

                 Centroid                   KNN                    NB  \
MI   [0.6791666666666667]  [0.7408333333333333]  [0.8241666666666667]   
IG                [0.815]  [0.8141666666666667]               [0.845]   
CHI              [0.8175]  [0.8058333333333333]  [0.8458333333333333]   
DF                 [0.82]  [0.8133333333333334]              [0.8425]   

                   Winnow                   SVM  
MI   [0.8041666666666667]  [0.8233333333333334]  
IG   [0.8366666666666667]  [0.8766666666666667]  
CHI  [0.8133333333333334]  [0.8658333333333333]  
DF   [0.8233333333333334]              [0.8775]  


In [7]:
micro_f1_table.head()

,Centroid,KNN,NB,Winnow,SVM
MI,[0.6791666666666667],[0.7408333333333333],[0.8241666666666667],[0.8041666666666667],[0.8233333333333334]
IG,[0.815],[0.8141666666666667],[0.845],[0.8366666666666667],[0.8766666666666667]
CHI,[0.8175],[0.8058333333333333],[0.8458333333333333],[0.8133333333333334],[0.8658333333333333]
DF,[0.82],[0.8133333333333334],[0.8425],[0.8233333333333334],[0.8775]


In [8]:
macro_f1_table.head()

,Centroid,KNN,NB,Winnow,SVM
MI,[0.6588268221195421],[0.738259368694502],[0.8241225751733594],[0.80372381853223],[0.8232228476130915]
IG,[0.8147730970438787],[0.8141655052010741],[0.844947885261435],[0.8366625832312474],[0.8766652962810697]
CHI,[0.8172196098042066],[0.8056064058112282],[0.8457946749150308],[0.8131043862064362],[0.8658287677844594]
DF,[0.8197997775305895],[0.8133203694701021],[0.8424202252390273],[0.8232842456237843],[0.8774978732269658]


In [9]:
# 将 micro_f1_table 中的列表转换为浮点数
micro_f1_table_float = micro_f1_table.applymap(lambda x: x[0] if isinstance(x, list) else x)

# 将 macro_f1_table 中的列表转换为浮点数
macro_f1_table_float = macro_f1_table.applymap(lambda x: x[0] if isinstance(x, list) else x)

print(micro_f1_table_float)
print(macro_f1_table_float)

     Centroid       KNN        NB    Winnow       SVM
MI   0.679167  0.740833  0.824167  0.804167  0.823333
IG   0.815000  0.814167  0.845000  0.836667  0.876667
CHI  0.817500  0.805833  0.845833  0.813333  0.865833
DF   0.820000  0.813333  0.842500  0.823333  0.877500
     Centroid       KNN        NB    Winnow       SVM
MI   0.658827  0.738259  0.824123  0.803724  0.823223
IG   0.814773  0.814166  0.844948  0.836663  0.876665
CHI  0.817220  0.805606  0.845795  0.813104  0.865829
DF   0.819800  0.813320  0.842420  0.823284  0.877498


C:\Users\xw\AppData\Local\Temp\ipykernel_3548\444534179.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  micro_f1_table_float = micro_f1_table.applymap(lambda x: x[0] if isinstance(x, list) else x)
C:\Users\xw\AppData\Local\Temp\ipykernel_3548\444534179.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  macro_f1_table_float = macro_f1_table.applymap(lambda x: x[0] if isinstance(x, list) else x)


In [10]:
# 计算 micro_f1_table 中每一行的平均值并添加为新列
micro_f1_table_float['mean'] = micro_f1_table_float.apply(lambda row: row.mean(), axis=1)

# 计算 macro_f1_table 中每一行的平均值并添加为新列
macro_f1_table_float['mean'] = macro_f1_table_float.apply(lambda row: row.mean(), axis=1)

# 打印更新后的 DataFrame
print(micro_f1_table_float)
print(macro_f1_table_float)

     Centroid       KNN        NB    Winnow       SVM      mean
MI   0.679167  0.740833  0.824167  0.804167  0.823333  0.774333
IG   0.815000  0.814167  0.845000  0.836667  0.876667  0.837500
CHI  0.817500  0.805833  0.845833  0.813333  0.865833  0.829667
DF   0.820000  0.813333  0.842500  0.823333  0.877500  0.835333
     Centroid       KNN        NB    Winnow       SVM      mean
MI   0.658827  0.738259  0.824123  0.803724  0.823223  0.769631
IG   0.814773  0.814166  0.844948  0.836663  0.876665  0.837443
CHI  0.817220  0.805606  0.845795  0.813104  0.865829  0.829511
DF   0.819800  0.813320  0.842420  0.823284  0.877498  0.835264


In [11]:
micro_column_means = micro_f1_table_float.mean()
macro_column_means = macro_f1_table_float.mean()

In [12]:
micro_f1_table_float.loc['mean'] = micro_f1_table_float.mean()
macro_f1_table_float.loc['mean'] = macro_f1_table_float.mean()
print(macro_f1_table_float)
print(micro_f1_table_float)

      Centroid       KNN        NB    Winnow       SVM      mean
MI    0.658827  0.738259  0.824123  0.803724  0.823223  0.769631
IG    0.814773  0.814166  0.844948  0.836663  0.876665  0.837443
CHI   0.817220  0.805606  0.845795  0.813104  0.865829  0.829511
DF    0.819800  0.813320  0.842420  0.823284  0.877498  0.835264
mean  0.777655  0.792838  0.839321  0.819194  0.860804  0.817962
      Centroid       KNN        NB    Winnow       SVM      mean
MI    0.679167  0.740833  0.824167  0.804167  0.823333  0.774333
IG    0.815000  0.814167  0.845000  0.836667  0.876667  0.837500
CHI   0.817500  0.805833  0.845833  0.813333  0.865833  0.829667
DF    0.820000  0.813333  0.842500  0.823333  0.877500  0.835333
mean  0.782917  0.793542  0.839375  0.819375  0.860833  0.819208


In [14]:
micro_f1_table_float.to_csv('micro_f1_table_float.csv')
macro_f1_table_float.to_csv('macro_f1_table_float.csv')